# 취업통계 2023–2024 OpenID 적용 검산

## tl;dr

- 개인정보 열을 제외한 안전 파생 패널 46,962행의 행 순서와 `_source_row_id`를 그대로 보존했다.
- 검토 승인된 482개 학교·연도 후보를 13,920행(29.6410%)에 적용했고, 242개 고유 OpenID가 연결됐다.
- 후보가 없는 33,042행은 공란으로 보존했다.
- 기존 비어 있지 않은 ID 덮어쓰기, 후보 충돌, 후보 키 누락, 행 ID 중복은 모두 0건이다.
- 적용값은 공식 EDSS 교차표가 아니라 승인된 추론 교차표이며, 방법·상태·출처 열을 함께 기록한다.


## Context & Methods

대학알리미 2023·2024년 재적학생수와 EDSS `0101`이 학교구분·지역·본분교 문맥 안에서 두 해 모두 정확히 같은 OpenID를 선택한 후보만 사용한다. 후보 파일의 학교·연도 키를 안전 파생 취업 패널의 동일 키에 결합하고, 기존 값이 있으면 보존하며 다른 값과 충돌하면 실행을 중단한다.

### Key Assumptions

- 사용자 검토 승인은 추론 후보를 별도 안전 파생 패널에 적용할 권한으로 해석한다.
- 원본 ZIP, 713MB 제한 패널, 후보 전용 기본 파생 패널은 수정하지 않는다.
- 공식 교차표가 아니므로 적용 근거 열을 제거하거나 공식 ID로 재표현하지 않는다.


## Data

- 기본 안전 파생 패널: `data/processed/edss/derived/employment_2023_2024_school_department.csv.gz`
- 검토 후보: `data/metadata/edss_employment_enrollment_open_id_candidates.csv`
- 적용 패널: `data/processed/edss/derived/employment_2023_2024_school_department_resolved.csv.gz`
- 감사 기록: `data/metadata/edss_employment_open_id_application.json`


In [1]:
import csv
import gzip
import json
from collections import Counter
from pathlib import Path

repo_root = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd().resolve()
source_path = repo_root / "data/processed/edss/derived/employment_2023_2024_school_department.csv.gz"
candidate_path = repo_root / "data/metadata/edss_employment_enrollment_open_id_candidates.csv"
resolved_path = repo_root / "data/processed/edss/derived/employment_2023_2024_school_department_resolved.csv.gz"
audit_path = repo_root / "data/metadata/edss_employment_open_id_application.json"

audit = json.loads(audit_path.read_text(encoding="utf-8"))
with gzip.open(source_path, "rt", encoding="utf-8", newline="") as handle:
    source_rows = list(csv.DictReader(handle))
with candidate_path.open(encoding="utf-8-sig", newline="") as handle:
    candidate_rows = list(csv.DictReader(handle))
with gzip.open(resolved_path, "rt", encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    resolved_fields = reader.fieldnames
    resolved_rows = list(reader)

print({
    "source_rows": len(source_rows),
    "candidate_school_years": len(candidate_rows),
    "resolved_rows": len(resolved_rows),
    "resolved_columns": len(resolved_fields),
})

{'source_rows': 46962, 'candidate_school_years': 482, 'resolved_rows': 46962, 'resolved_columns': 36}


## Results

In [2]:
candidate_lookup = {
    (row["_panel_year"], row["_school_identity_key"]): row["candidate_open_id"]
    for row in candidate_rows
}
applied_rows = [
    row for row in resolved_rows
    if row["_open_id_resolution_status"] == "applied_reviewed_inferred_crosswalk"
]
unresolved_rows = [row for row in resolved_rows if not row["개방ID"]]

assert len(source_rows) == len(resolved_rows) == 46_962
assert [row["_source_row_id"] for row in source_rows] == [
    row["_source_row_id"] for row in resolved_rows
]
assert len({row["_source_row_id"] for row in resolved_rows}) == 46_962
assert len(candidate_lookup) == 482
assert len({(row["_panel_year"], row["candidate_open_id"]) for row in candidate_rows}) == 482
assert len(applied_rows) == 13_920
assert len(unresolved_rows) == 33_042
assert all(
    row["개방ID"] == candidate_lookup[(row["_panel_year"], row["_school_identity_key"])]
    for row in applied_rows
)
assert audit["application"]["conflict_count"] == 0
assert audit["application"]["overwritten_nonempty_open_id_row_count"] == 0
assert audit["application"]["unmatched_candidate_key_count"] == 0

results = {
    "row_coverage_pct": round(100 * len(applied_rows) / len(resolved_rows), 4),
    "identity_coverage_pct": round(100 * 482 / 3281, 4),
    "applied_rows_by_year": dict(Counter(row["_panel_year"] for row in applied_rows)),
    "applied_distinct_open_ids": len({row["개방ID"] for row in applied_rows}),
    "remaining_missing_rows": len(unresolved_rows),
    "conflicts": audit["application"]["conflict_count"],
    "overwrites": audit["application"]["overwritten_nonempty_open_id_row_count"],
}
results

{'row_coverage_pct': 29.641,
 'identity_coverage_pct': 14.6906,
 'applied_rows_by_year': {'2023': 6895, '2024': 7025},
 'applied_distinct_open_ids': 242,
 'remaining_missing_rows': 33042,
 'conflicts': 0,
 'overwrites': 0}

In [3]:
assert results == {
    "row_coverage_pct": 29.641,
    "identity_coverage_pct": 14.6906,
    "applied_rows_by_year": {"2023": 6895, "2024": 7025},
    "applied_distinct_open_ids": 242,
    "remaining_missing_rows": 33042,
    "conflicts": 0,
    "overwrites": 0,
}
print("All application invariants passed.")
print("Output SHA-256:", audit["output"]["sha256"])

All application invariants passed.
Output SHA-256: 662d76392e62d2cf45735119d37cde6aefeb1c60eae3ef71eea064e299e22fd3


## Takeaways

- 적용 패널은 원본과 동일한 46,962행을 유지하므로 결합 과정의 행 손실·증식이 없다.
- 13,920행은 승인된 후보로 분석 연결이 가능하며 33,042행은 계속 미해결로 분리해야 한다.
- `개방ID`만 보고 공식 교차표라고 해석하지 말고 `_open_id_resolution_*` 열을 함께 사용해야 한다.
- 후보가 추가되면 동일 승인 플래그로 스크립트를 재실행하고 이 노트북의 불변식을 다시 확인한다.
